# E8a: MCT-LTDiag on all 203 patients (E8.0) and radiomics mitigation (E8a)

**Revision plan, 00 log section 21 (approved 17 Oct 2026).** Answers Reviewer 2 point 6 ("characterises failures without
mitigation") and strengthens R4-2, R2-3 and R2-4. Every method uses internal data only or MCT-LTDiag images without
their labels; each is evaluated once; all results are reported; **the E6 primary result stays the primary result.**

**E8.0 — E6 on all 203 patients.** E6 excluded 2 HCC patients because its facts check crashed when their tumour and
liver masks were on different grids (a code error; rule 17.3-2 says such masks are resampled). This notebook runs **E6's
own cells** for those 2 patients with that one line fixed, merges them with the E6 run, and recomputes E6's results on
203 patients. A check confirms the merged data reproduce E6's saved values for the other 201.

**E8a — radiomics mitigation (MCT-LTDiag, primary preprocessing):**
1. **Reference-batch ComBat** (neuroCombat): MCT-LTDiag features mapped onto the internal development distribution; internal
   features unchanged, so the frozen E2 models apply. No labels, no covariates.
2. **Perturbation-robust features:** internal features recomputed under 5 small perturbations (1-voxel shift, 5° in-plane
   rotation, noise σ = 2, mask erosion, mask dilation); features with ICC(1,1) ≥ 0.90 kept; the E2 recipe retrained on
   them (same folds and seed).
3. Robust features + ComBat.
Each radiomics model is also used in CNN + radiomics and full fusion with the phase-dropout CNN (meta-learners fitted on
internal out-of-fold predictions only). HCC-TACE-Seg (corrected, E5c) is scored with the robust model; ComBat is not
applied there because that cohort is HCC-only (harmonising it would remove disease signal).

**Runtime:** GPU runtime (L4) for its CPUs; about 1–1.5 hours. **Safety:** inputs are only read; outputs go to a new
folder `liver_project/revision_2026-10/E8a_<time>/`.

In [ ]:
# Cell 1 — Setup (E6's cell): installs, paths, input checks, output folder E8a_<time>
import os, sys, io, re, json, time, shutil, tarfile, hashlib, platform, datetime, warnings, subprocess
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    WORK = Path('/content/e6_work')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK', 'monai', 'lightgbm', 'pydicom', 'nibabel', 'neuroCombat'], check=True)
    radiomics_ok = False
    for spec in (['pyradiomics'], ['git+https://github.com/AIM-Harvard/pyradiomics.git']):
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + spec).returncode == 0:
            radiomics_ok = True
            break
    if not radiomics_ok:
        raise RuntimeError('PyRadiomics could not be installed. Send this output for review.')
else:
    ROOT = Path(os.environ['E6_TEST_ROOT'])
    WORK = Path(os.environ['E6_TEST_WORK'])
WORK.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import SimpleITK as sitk
import nibabel as nib
import pydicom
import lightgbm as lgb
from neuroCombat import neuroCombat
from sklearn.feature_selection import f_classif
import joblib
from radiomics import featureextractor
from monai.networks.nets import resnet18
from scipy import ndimage
from scipy.stats import beta, norm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
MCT_DIR = PROJECT_DIR / 'external_validation' / 'MCT-LTDiag'
E6_RUN = REVISION_DIR / 'E6_20261017T114646Z'
E5C_RUN = REVISION_DIR / 'E5c_20261017T143217Z'
RESTORE = None  # derived in Cell 3: the selected patients missing from the saved E6 predictions
SEED = 42
CLASSES = ['HCC', 'ICC', 'CHCC']
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']
N_IO = 6 if IN_COLAB else 2                       # patients read/preprocessed at the same time
N_JOBS = os.cpu_count() if IN_COLAB else 1        # parallel radiomics processes

INPUTS = {
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_noshape_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_original_predictions': (E3_DIR / 'E3A_original_cnn_predictions.csv', '5c02c3ebda984b6283c1137d85c17c171347b24ae1fac450dfc89c5513d0229b'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'mct_selected': (MCT_DIR / 'metadata' / 'selected_HCC_ICC_patients.csv', 'fdb8d8fa74d8211bbfe45ecb4820415e52a5183ce5d51bcfa4376cfa9ed11e20'),
    'mct_patient_table': (MCT_DIR / 'metadata' / 'meta_info_patient.csv', 'f9f0fc730b1ec75cecb615304aa0f69db30225fb559b460ef78d0fe693220f0d'),
    'e6_predictions': (E6_RUN / 'E6_predictions.csv', 'c394f78e9972609f3bea237d464e25cab2d547e4b2ef191fa4c6676013f65698'),
    'e6_orientation': (E6_RUN / 'E6_orientation_decisions.csv', 'a1342374d4c5b1b575e3f06359b6aa2240349e085c4045a4fd97f89cf795011a'),
    'e6_rad_primary': (E6_RUN / 'E6_radiomics_features_primary.csv', 'ac7077085e8f9e4bc458c14fe6a0cd8ba109d23898a81d8a3d70e289b8db7f56'),
    'e6_rad_S1_native': (E6_RUN / 'E6_radiomics_features_S1_native.csv', '64bdab577160097bb54ed6fa38b79cfd20aa35305a637915acf123efd1bc4a2a'),
    'e6_rad_S2_iso1mm': (E6_RUN / 'E6_radiomics_features_S2_iso1mm.csv', '2fdc08f594f895581f9deed9f724bc2300ad20f6d8f5d1185c58058e56838903'),
    'e6_rad_S4_reversed_slices': (E6_RUN / 'E6_radiomics_features_S4_reversed_slices.csv', 'ac1f73a999fef82f434bd74435aede7f3e8722546b4532301a8b9807e17ffe06'),
    'e6_rad_S5_week10g_axes': (E6_RUN / 'E6_radiomics_features_S5_week10g_axes.csv', 'c2bb39b3dede33136d54377a5dab4f17830a1ec28b939990c56ae1788b87a417'),
    'e2_radiomics_oof': (E2_DIR / 'E2_radiomics_noshape_oof_predictions.csv', '21057b42933293a7d45cbefbb6f58dfe89458064a42fff689abcbc826fe28cb6'),
    'e2_test_meta': (E2_DIR / 'E2_fusion_test_meta_features_noshape.csv', 'fbb946f641a050f66b093564141a8e103e1c4e5c651d6abeb51042293ab8cb2f'),
    'e5c_predictions': (E5C_RUN / 'E5c_predictions.csv', '45dd7f05598ba5cff5fb57091622d36afd332f55bf1b29a5e216eec0ed6999de'),
    'e5c_rad_E6rules': (E5C_RUN / 'E5c_radiomics_features_axis_fixed_E6rules.csv', 'b191f36146407e3d980e25ef3b3c72c4478fb90aee6f241e2c424809add4f18b'),
    'e5c_mixed': (E5C_RUN / 'E5_mixed_scan_inputs.csv', '4a26216e8f03afade00ff4608f821965aba7df46074a0aae6644216bb6c19155'),
    'e5c_clinical': (E5C_RUN / 'E5_external_clinical_joined.csv', '55617486a87ecc84dcbcd71404ef36dcdcf91a9ca2678c4d10d02452c2c4c75e'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
LARGE_INPUTS = {'processed_tar': (PROJECT_DIR / 'processed.tar', 402616320)}
MODEL_FILES = {
    **{f'w3_{f}': PROJECT_DIR / 'checkpoints_w3' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'w4_{f}': PROJECT_DIR / 'checkpoints_w4' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)},
    **{f'rad_{f}': E2_DIR / 'e2_radiomics_noshape_fold_models' / f'fold_{f}.joblib' for f in range(5)},
}

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

def md5_of(path):
    digest = hashlib.md5()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 24), b''):
            digest.update(block)
    return digest.hexdigest()

problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
for key, (path, size) in LARGE_INPUTS.items():
    if not path.exists() or (IN_COLAB and path.stat().st_size != size):
        problems.append(f'missing or changed size: {path}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import sklearn, scipy, radiomics, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version_VersionString() if hasattr(sitk, 'Version_VersionString') else 'unknown',
            'pyradiomics': getattr(radiomics, '__version__', 'unknown'), 'pydicom': getattr(pydicom, '__version__', 'unknown'),
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'device': str(DEVICE), 'cpus': os.cpu_count()}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'E8a_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=False)
print('Inputs and models checked:', len(INPUTS) + len(LARGE_INPUTS) + len(MODEL_FILES))
print('Versions:', VERSIONS)
print('Output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Frozen models must reproduce their saved internal test predictions; internal orientation reference
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(LARGE_INPUTS['processed_tar'][0]), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E6_TEST_PROCESSED'])

split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
TEST_CT, TEST_LABEL = {}, {}
for pid in test_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        TEST_CT[pid], TEST_LABEL[pid] = z['ct'].astype(np.uint8), int(z['label'])

def zscore_nonzero_per_phase(x4):          # Week 4, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def make_cnn(in_channels):
    return resnet18(spatial_dims=3, n_input_channels=in_channels, num_classes=3, feed_forward=True)

def load_models(keys, in_channels):
    models = []
    for key in keys:
        checkpoint = torch.load(MODEL_FILES[key], map_location=DEVICE, weights_only=False)
        model = make_cnn(in_channels).to(DEVICE)
        model.load_state_dict(checkpoint['state_dict'])
        model.eval()
        models.append(model)
    return models

CNNS = {'single_phase_CNN': (load_models([f'w3_{f}' for f in range(5)], 1), [2]),
        'original_CNN': (load_models([f'w4_{f}' for f in range(5)], 4), [0, 1, 2, 3]),
        'phase_dropout_CNN': (load_models([f'd_{f}' for f in range(5)], 4), [0, 1, 2, 3])}

@torch.no_grad()
def cnn_probs(models, channels, arrays):
    """Mean softmax over the 5 fold models; arrays: patient -> uint8 (4, 96, 96, 96) in internal axis order."""
    rows = {}
    for pid, ct in arrays.items():
        x = torch.from_numpy(zscore_nonzero_per_phase(ct)[channels]).unsqueeze(0).to(DEVICE)
        rows[pid] = np.mean([torch.softmax(m(x), dim=1).float().cpu().numpy()[0] for m in models], axis=0)
    return pd.DataFrame.from_dict(rows, orient='index', columns=['p_HCC', 'p_ICC', 'p_CHCC'])

def macro_auc3(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))

y_test = [TEST_LABEL[p] for p in test_ids]
gate = {}
for name, key, model_tag in [('original_CNN', 'e3_original_predictions', 'original'),
                             ('phase_dropout_CNN', 'e3_d_predictions', 'D_phase_dropout')]:
    saved = pd.read_csv(INPUTS[key][0], dtype={'patient_id': str})
    saved = saved.query("pattern == 'complete' and split == 'test'").set_index('patient_id').loc[test_ids, ['p_HCC', 'p_ICC', 'p_CHCC']]
    ours = cnn_probs(*CNNS[name], {p: TEST_CT[p] for p in test_ids}).loc[test_ids]
    gate[name] = {'max_prob_diff': float(np.abs(ours.to_numpy() - saved.to_numpy()).max()),
                  'macro_auc_ours': macro_auc3(y_test, ours.to_numpy()), 'macro_auc_saved': macro_auc3(y_test, saved.to_numpy())}

internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
rad_models = [joblib.load(MODEL_FILES[f'rad_{f}']) for f in range(5)]
pool_cols = rad_models[0]['feature_pool']
X_int_test = internal.set_index('patient_id').loc[test_ids, pool_cols].to_numpy(dtype=np.float32)
p_int = np.mean([m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X_int_test[:, m['sel_idx']])))
                 for m in rad_models], axis=0)
saved_int = pd.read_csv(INPUTS['e2_noshape_test'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids, ['p_HCC', 'p_ICC', 'p_CHCC']].to_numpy()
gate['radiomics_noshape'] = {'max_prob_diff': float(np.abs(p_int - saved_int).max())}
GATE_OK = (all(gate[n]['max_prob_diff'] < 0.02 and abs(gate[n]['macro_auc_ours'] - gate[n]['macro_auc_saved']) < 0.005
               for n in ['original_CNN', 'phase_dropout_CNN']) and gate['radiomics_noshape']['max_prob_diff'] < 1e-6)
print('Internal reproduction check:', json.dumps(gate, indent=1))
if IN_COLAB and not GATE_OK:
    raise RuntimeError('STOP — the frozen models do not reproduce their saved internal test predictions.')

# ---- internal orientation reference (stored arrays, as Week 2 read them with nibabel) ----
def load_nii(path):                          # Week 2, unchanged
    arr = nib.load(str(path)).get_fdata().astype(np.float32)
    while arr.ndim > 3:
        if arr.shape[-1] == 1:
            arr = arr.squeeze(-1)
        elif arr.shape[-1] == 3:
            arr = arr.mean(axis=-1)
        else:
            arr = arr[..., 0]
    return arr

BONE_0255 = 254                               # >= ~198 HU on the -200..200 -> 0..255 scale

def orientation_descriptor(ct0255_ijk, liver_ijk):
    """Where the liver and the bone are in an (i, j, k) array, as fractions of each axis (no class labels)."""
    li, lj, lk = np.nonzero(liver_ijk)
    if li.size < 500:
        return None
    shape = liver_ijk.shape
    bone = (ct0255_ijk >= BONE_0255) & ~liver_ijk
    bi, bj, _ = np.nonzero(bone)
    area = liver_ijk.sum(axis=(0, 1)).astype(np.float64)
    k0, k1 = int(lk.min()), int(lk.max())
    centre_k = float((np.arange(len(area)) * area).sum() / area.sum())
    return {'liver_i': float(li.mean() / shape[0]), 'liver_j': float(lj.mean() / shape[1]),
            'bone_i': float(bi.mean() / shape[0]) if bi.size >= 200 else np.nan,
            'bone_j': float(bj.mean() / shape[1]) if bj.size >= 200 else np.nan,
            'bulk_k': (centre_k - k0) / max(1, k1 - k0)}

patients_table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
p_rows = patients_table[patients_table['phase'] == 'P'].drop_duplicates('patient_id')   # non-contrast: bone not confused with contrast

def internal_descriptor(row):
    try:
        return orientation_descriptor(load_nii(DATA_DIR / row['ct_path']), load_nii(DATA_DIR / row['liver_mask_path']) > 0)
    except Exception as error:
        return {'error': repr(error)[:120]}

with ThreadPoolExecutor(8 if IN_COLAB else 2) as pool:
    internal_desc = pd.DataFrame([d or {} for d in pool.map(internal_descriptor, [r for _, r in p_rows.iterrows()])])
internal_desc.insert(0, 'patient_id', p_rows['patient_id'].tolist())
internal_desc.to_csv(OUT_DIR / 'E8_0_internal_orientation_reference.csv', index=False)
ok = internal_desc.dropna(subset=['liver_i', 'bone_i', 'bone_j'])
REF = {'front_back_axis_is_j': float((abs(ok['bone_j'] - .5) > abs(ok['bone_i'] - .5)).mean()),
       'liver_at_low_i': float((ok['liver_i'] < .5).mean()), 'bone_at_high_j': float((ok['bone_j'] > .5).mean()),
       'liver_bulk_at_low_k': float((ok['bulk_k'] < .5).mean()), 'n': int(len(ok)),
       'median_liver_i': float(ok['liver_i'].median()), 'median_bone_j': float(ok['bone_j'].median()),
       'median_bulk_k': float(ok['bulk_k'].median())}
print('Internal orientation reference:', json.dumps(REF, indent=1))
# Must agree with the 12-patient check in the 00 log (17.9): i = left-right with the liver at low i, spine at high j, bulk at low k
if not (REF['front_back_axis_is_j'] >= .8 and REF['liver_at_low_i'] >= .8 and REF['bone_at_high_j'] >= .8 and REF['liver_bulk_at_low_k'] >= .7):
    raise RuntimeError('STOP — the internal orientation reference differs from the 17 Oct check. Send this output.')

In [ ]:
# Cell 3 — E8.0: E6's facts check for the 2 excluded patients only, with the off-grid fix (rule 17.3-2)
SEL = pd.read_csv(INPUTS['mct_selected'][0], dtype=str)
SEL['size'] = SEL['size'].astype(int)
counts = SEL['type'].value_counts().to_dict()
if IN_COLAB and counts != {'HCC': 103, 'ICC': 100}:
    raise RuntimeError(f'STOP — unexpected MCT-LTDiag selection {counts}.')
WANTED = {'NIFTI/nc.nii.gz': 'P', 'NIFTI/art.nii.gz': 'C1', 'NIFTI/pvp.nii.gz': 'C2', 'NIFTI/delay.nii.gz': 'C3',
          'mask_pvp.nii.gz': 'tumor', 'liver_mask_pvp.nii.gz': 'liver'}
DICOM_FIELDS = ['SeriesDescription', 'Manufacturer', 'ManufacturerModelName', 'SliceThickness', 'PixelSpacing', 'KVP',
                'ConvolutionKernel', 'SeriesInstanceUID']
TAR_TMP, NII_DIR = WORK / 'tar', WORK / 'nifti'
TAR_TMP.mkdir(exist_ok=True)
NII_DIR.mkdir(exist_ok=True)

def window_0255(hu):                          # PLC-CECT release convention: -200..200 HU -> 0..255
    return ((np.clip(hu, -200.0, 200.0) + 200.0) / 400.0 * 255.0).astype(np.float32)

def ijk(img):                                 # SimpleITK array (k, j, i) -> stored (i, j, k) order, as nibabel / Week 2
    arr = sitk.GetArrayFromImage(img)
    return np.ascontiguousarray(np.transpose(arr, (2, 1, 0)))

def grid(img):
    return img.GetSize(), img.GetSpacing(), img.GetOrigin(), img.GetDirection()

def same_grid(a, b):
    return (tuple(a[0]) == tuple(b[0]) and np.allclose(a[1], b[1], rtol=1e-3) and np.allclose(a[2], b[2], atol=0.5)
            and np.allclose(a[3], b[3], atol=1e-3))

def onto(img, reference, is_mask):
    return sitk.Resample(img, reference, sitk.Transform(), sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear,
                         0 if is_mask else -1024)

def nifti_facts(folder):
    imgs = {k: sitk.ReadImage(str(folder / f'{k}.nii.gz')) for k in PHASE_ORDER + ['tumor', 'liver']}
    ref = grid(imgs['C2'])
    f = {'size': list(ref[0]), 'spacing': [round(float(s), 4) for s in ref[1]], 'direction': [round(float(d), 3) for d in ref[3]],
         'off_grid': [k for k in imgs if k != 'C2' and not same_grid(grid(imgs[k]), ref)]}
    mm = sitk.MinimumMaximumImageFilter()
    for k in PHASE_ORDER:
        mm.Execute(imgs[k])
        sample = sitk.GetArrayViewFromImage(imgs[k])[::2, ::4, ::4]
        f[f'{k}_min'], f[f'{k}_max'] = float(mm.GetMinimum()), float(mm.GetMaximum())
        f[f'{k}_p50'] = float(np.percentile(sample, 50))
    mins, maxs = [f[f'{k}_min'] for k in PHASE_ORDER], [f[f'{k}_max'] for k in PHASE_ORDER]
    f['intensity_kind'] = ('HU' if max(mins) <= -500 else '0-255' if min(mins) >= 0 and max(maxs) <= 255 else 'unexpected')
    # E8.0 fix: masks on another grid are resampled onto the portal-venous grid before they are compared
    tumor_img = imgs['tumor'] if 'tumor' not in f['off_grid'] else onto(imgs['tumor'], imgs['C2'], is_mask=True)
    liver_img0 = imgs['liver'] if 'liver' not in f['off_grid'] else onto(imgs['liver'], imgs['C2'], is_mask=True)
    tumor, liver = sitk.GetArrayFromImage(tumor_img), sitk.GetArrayFromImage(liver_img0)
    f['tumor_labels'] = [float(v) for v in np.unique(tumor)[:10]]
    f['liver_labels'] = [float(v) for v in np.unique(liver)[:10]]
    f['tumor_voxels'], f['liver_voxels'] = int((tumor > 0).sum()), int((liver > 0).sum())
    f['tumor_components'] = int(ndimage.label(tumor > 0)[1])
    f['tumor_volume_ml'] = f['tumor_voxels'] * float(np.prod(ref[1])) / 1000.0
    f['tumor_outside_liver_fraction'] = float(((tumor > 0) & (liver == 0)).sum() / max(1, (tumor > 0).sum()))
    plain = imgs['P'] if 'P' not in f['off_grid'] else onto(imgs['P'], imgs['C2'], is_mask=False)
    liver_img = imgs['liver'] if 'liver' not in f['off_grid'] else onto(imgs['liver'], imgs['C2'], is_mask=True)
    ct = ijk(plain)
    descriptor = orientation_descriptor(window_0255(ct) if f['intensity_kind'] == 'HU' else ct, ijk(liver_img) > 0)
    if descriptor is None or not np.isfinite(descriptor['bone_i']):
        raise RuntimeError('liver mask or bone too small to determine orientation')
    f.update(descriptor)
    f['nib_equals_sitk_ijk'] = bool(np.allclose(load_nii(folder / 'P.nii.gz'), ijk(imgs['P']).astype(np.float32), atol=1e-3))
    return f

def stage(row):
    pid = row['patient_id']
    folder = NII_DIR / pid
    done = folder / 'facts.json'
    if done.exists():
        return json.loads(done.read_text())
    folder.mkdir(exist_ok=True)
    facts, series = {'patient_id': pid, 'technical_failure': ''}, {}
    local = TAR_TMP / row['name']
    try:
        shutil.copyfile(MCT_DIR / f"tar_{row['type']}" / row['name'], local)
        if md5_of(local) != row['md5']:
            raise RuntimeError('checksum mismatch after copying from Drive')
        found = set()
        with tarfile.open(local, 'r:') as tf:
            for m in tf:
                if not m.isfile():
                    continue
                name = m.name[2:] if m.name.startswith('./') else m.name
                if name in WANTED:
                    (folder / f'{WANTED[name]}.nii.gz').write_bytes(tf.extractfile(m).read())   # fixed local names only
                    found.add(name)
                elif name.startswith('DICOM/'):
                    key = name.rsplit('/', 1)[0]
                    if key not in series:
                        try:
                            ds = pydicom.dcmread(io.BytesIO(tf.extractfile(m).read()), stop_before_pixels=True, force=True)
                            series[key] = {field: str(getattr(ds, field, '')) for field in DICOM_FIELDS}
                        except Exception as error:
                            series[key] = {'error': repr(error)[:100]}
                        series[key]['files'] = 0
                    series[key]['files'] += 1
        if set(WANTED) - found:
            raise RuntimeError(f'missing in tar: {sorted(set(WANTED) - found)}')
        facts.update(nifti_facts(folder))
        if facts['tumor_voxels'] == 0:
            facts['technical_failure'] = 'empty tumour mask'
    except Exception as error:
        facts['technical_failure'] = repr(error)[:200]
    finally:
        local.unlink(missing_ok=True)
    facts['dicom_series'] = list(series.values())
    done.write_text(json.dumps(facts, default=float))
    return facts

t0 = time.time()
# the 2 patients excluded by the E6 facts-check code error = selected patients absent from E6's predictions
_e6_ids = set(pd.read_csv(INPUTS['e6_predictions'][0], dtype={'patient_id': str})['patient_id'])
RESTORE = sorted(set(SEL['patient_id']) - _e6_ids)
rows = [r for _, r in SEL.iterrows() if r['patient_id'] in RESTORE]
FACTS = []
with ThreadPoolExecutor(N_IO) as pool:
    for n, f in enumerate(pool.map(stage, rows), start=1):
        FACTS.append(f)
        if n % 20 == 0 or n == len(rows):
            print(f'  facts check {n}/{len(rows)} ({(time.time() - t0) / 60:.1f} min)', flush=True)

dicom_rows = [{'patient_id': f['patient_id'], **s} for f in FACTS for s in f.get('dicom_series', [])]
pd.DataFrame(dicom_rows).to_csv(OUT_DIR / 'E8_0_restored2_dicom_series.csv', index=False)
facts_df = pd.DataFrame([{k: v for k, v in f.items() if k != 'dicom_series'} for f in FACTS])
TECH_FAIL = facts_df.loc[facts_df['technical_failure'] != '', ['patient_id', 'technical_failure']]
INCLUDED = facts_df.loc[facts_df['technical_failure'] == '', 'patient_id'].tolist()
kinds = Counter(facts_df.loc[facts_df['technical_failure'] == '', 'intensity_kind'])
print(f'Restored {len(INCLUDED)}/{len(RESTORE)}; technical failures: {TECH_FAIL.to_dict("records")}')
if sorted(INCLUDED) != sorted(RESTORE):
    raise RuntimeError('STOP — the 2 excluded patients could not be restored. Send this output.')
print('Intensity kind:', dict(kinds), '| phases off the portal-venous grid:',
      int(facts_df['off_grid'].map(lambda v: len(v) if isinstance(v, list) else 0).gt(0).sum()), 'patients')
if kinds.get('unexpected', 0):
    raise RuntimeError('STOP — some scans are neither HU nor 0-255 (rule 17.3-1). Send this output.')
if IN_COLAB and not facts_df.loc[facts_df['technical_failure'] == '', 'nib_equals_sitk_ijk'].all():
    raise RuntimeError('STOP — SimpleITK (i, j, k) arrays differ from nibabel arrays.')

# ---- orientation decisions (amendment 17.9-1): match each scan to the internal anatomy ----
def decide(d):
    # front-back axis = the in-plane axis on which the bone (spine) is off-centre; internally that is j
    off_i, off_j = abs(d['bone_i'] - .5), abs(d['bone_j'] - .5)
    out = {'transpose': off_i > off_j, 'conf_transpose': max(off_i, off_j) >= .05 and abs(off_i - off_j) >= .03}
    li, bj = (d['liver_j'], d['bone_i']) if out['transpose'] else (d['liver_i'], d['bone_j'])
    out['flip_i'], out['conf_flip_i'] = bool(li > .5), abs(li - .5) >= .05
    out['flip_j'] = bool(bj < .5) if np.isfinite(bj) else False
    out['conf_flip_j'] = bool(np.isfinite(bj) and abs(bj - .5) >= .05)
    out['flip_k'], out['conf_flip_k'] = bool(d['bulk_k'] > .5), abs(d['bulk_k'] - .5) >= .03
    return out

ORIENT = {f['patient_id']: decide(f) for f in FACTS if f['patient_id'] in INCLUDED}
orient_df = pd.DataFrame.from_dict(ORIENT, orient='index')
E6_DECISIONS = pd.read_csv(INPUTS['e6_orientation'][0], index_col=0)
E6_MAJORITY = {part: bool(E6_DECISIONS[part].astype(bool).mean() > .5) for part in ['transpose', 'flip_i', 'flip_j', 'flip_k']}
orientation_summary = {}
for part in ['transpose', 'flip_i', 'flip_j', 'flip_k']:
    unclear = orient_df.index[~orient_df[f'conf_{part}']]
    orient_df.loc[unclear, part] = E6_MAJORITY[part]                 # the same fallback the E6 run used
    orientation_summary[part] = {'applied_to': int(orient_df[part].sum()), 'unclear_set_to_E6_majority': int(len(unclear)),
                                 'E6_majority': E6_MAJORITY[part]}
ORIENT = orient_df[['transpose', 'flip_i', 'flip_j', 'flip_k']].astype(bool).to_dict('index')
orient_df.to_csv(OUT_DIR / 'E8_0_restored2_orientation_decisions.csv')
facts_df.to_csv(OUT_DIR / 'E8_0_restored2_facts.csv', index=False)

ok = facts_df[facts_df['technical_failure'] == '']
dicom = pd.DataFrame(dicom_rows)
FACTS_SUMMARY = {
    'in_plane_spacing_mm': [float(ok['spacing'].map(lambda s: s[0]).min()), float(ok['spacing'].map(lambda s: s[0]).median()),
                            float(ok['spacing'].map(lambda s: s[0]).max())],
    'slice_spacing_mm': [float(ok['spacing'].map(lambda s: s[2]).min()), float(ok['spacing'].map(lambda s: s[2]).median()),
                         float(ok['spacing'].map(lambda s: s[2]).max())],
    'tumour_components': ok['tumor_components'].value_counts().sort_index().to_dict(),
    'tumour_volume_ml_median': float(ok['tumor_volume_ml'].median()),
    'manufacturers': dicom['Manufacturer'].value_counts().to_dict() if 'Manufacturer' in dicom else {},
    'dicom_slice_thickness': dicom['SliceThickness'].value_counts().head(6).to_dict() if 'SliceThickness' in dicom else {},
    'kvp': dicom['KVP'].value_counts().head(6).to_dict() if 'KVP' in dicom else {},
}
print('Facts:', json.dumps(FACTS_SUMMARY, indent=1, default=str))
print('Orientation decisions (amendment 17.9-1):', json.dumps(orientation_summary, indent=1))


def header_orientation(direction):            # amendment 18.1 rule, used here only as a cross-check
    D = np.array(direction, dtype=float).reshape(3, 3)
    p = [int(np.argmax(np.abs(D[:, c]))) for c in range(3)]
    s = [float(np.sign(D[p[c], c])) for c in range(3)]
    if p[2] != 2 or sorted(p[:2]) != [0, 1]:
        return None
    transpose = p[0] == 1
    a0, a1 = (1, 0) if transpose else (0, 1)
    return {'transpose': bool(transpose), 'flip_i': bool(s[a0] < 0), 'flip_j': bool(s[a1] < 0), 'flip_k': bool(s[2] > 0)}

orientation_summary['header_agrees'] = {f['patient_id']: header_orientation(f['direction']) == ORIENT[f['patient_id']]
                                        for f in FACTS if f['patient_id'] in ORIENT}
print('Header cross-check (True = same decision):', orientation_summary['header_agrees'])

In [ ]:
# Cell 4 — Preprocessing: primary and sensitivity variants (internal Week 2 crop rules)
INPLANE_MM = 400.0 / 512                       # internal physical pixel size (512 matrix, 400 mm field of view)
CROP_SIZE = 96
VARIANTS = ['primary', 'S1_native', 'S2_iso1mm', 'S4_reversed_slices', 'S5_week10g_axes']
CROP_DIR = WORK / 'crops'
CROP_DIR.mkdir(exist_ok=True)

def compute_center_of_mass(mask):             # Week 2, unchanged
    coords = np.array(np.where(mask > 0))
    if coords.shape[1] == 0:
        return None
    return tuple(int(round(c)) for c in coords.mean(axis=1))

def crop_cube(volume, center, size):          # Week 2, unchanged
    half = size // 2
    shape = volume.shape
    starts = [c - half for c in center]
    ends = [c + size - half for c in center]
    pad_before = [max(0, -s) for s in starts]
    pad_after = [max(0, e - shape[i]) for i, e in enumerate(ends)]
    s0, s1, s2 = (max(0, s) for s in starts)
    e0, e1, e2 = (min(shape[i], e) for i, e in enumerate(ends))
    cropped = volume[s0:e0, s1:e1, s2:e2]
    if any(pad_before) or any(pad_after):
        cropped = np.pad(cropped, [(pad_before[i], pad_after[i]) for i in range(3)], mode='constant', constant_values=0)
    return cropped

def resample(img, spacing, is_mask):
    size = [max(1, int(round(sz * sp / ts))) for sz, sp, ts in zip(img.GetSize(), img.GetSpacing(), spacing)]
    f = sitk.ResampleImageFilter()
    f.SetOutputSpacing(tuple(float(s) for s in spacing))
    f.SetSize(size)
    f.SetOutputOrigin(img.GetOrigin())
    f.SetOutputDirection(img.GetDirection())
    f.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    f.SetDefaultPixelValue(0 if is_mask else -1024)
    return f.Execute(img)

def apply_orientation(a, o):
    if o['transpose']:
        a = np.transpose(a, (1, 0, 2))
    if o['flip_i']:
        a = a[::-1]
    if o['flip_j']:
        a = a[:, ::-1]
    if o['flip_k']:
        a = a[:, :, ::-1]
    return np.ascontiguousarray(a)

def crops_from(arrays):
    liver, tumor = arrays['liver'] > 0, arrays['tumor'] > 0
    center = compute_center_of_mass(tumor) or compute_center_of_mass(liver)
    ct4 = np.stack([np.clip(crop_cube(arrays[p] * liver, center, CROP_SIZE), 0, 255).astype(np.uint8) for p in PHASE_ORDER])
    tumor_crop = (crop_cube(tumor.astype(np.uint8), center, CROP_SIZE) > 0).astype(np.uint8)
    meta = {'shape': list(tumor.shape), 'center': list(center), 'tumor_in_crop': float(tumor_crop.sum() / max(1, tumor.sum()))}
    return ct4, tumor_crop, meta

def preprocess(pid):
    target = CROP_DIR / f'{pid}.npz'
    meta_file = CROP_DIR / f'{pid}.json'
    if target.exists() and meta_file.exists():
        return json.loads(meta_file.read_text())
    folder = NII_DIR / pid
    facts = next(f for f in FACTS if f['patient_id'] == pid)
    imgs = {k: sitk.ReadImage(str(folder / f'{k}.nii.gz')) for k in PHASE_ORDER + ['tumor', 'liver']}
    for k in facts['off_grid']:                                  # rule 17.3-2
        imgs[k] = onto(imgs[k], imgs['C2'], is_mask=k in ('tumor', 'liver'))
    hu = facts['intensity_kind'] == 'HU'
    sz = imgs['C2'].GetSpacing()[2]
    settings = {'primary': (INPLANE_MM, INPLANE_MM, sz), 'S1_native': None, 'S2_iso1mm': (1.0, 1.0, 1.0),
                'S5_week10g_axes': (1.0, 1.0, 1.0)}
    out, meta = {}, {}
    for variant, spacing in settings.items():
        vols = imgs if spacing is None else {k: resample(v, spacing, k in ('tumor', 'liver')) for k, v in imgs.items()}
        if variant == 'S5_week10g_axes':
            arrays = {k: sitk.GetArrayFromImage(v).astype(np.float32) for k, v in vols.items()}   # (k, j, i), as Week 10g
        else:
            arrays = {k: apply_orientation(ijk(v).astype(np.float32), ORIENT[pid]) for k, v in vols.items()}
        for p in PHASE_ORDER:
            arrays[p] = window_0255(arrays[p]) if hu else arrays[p]
        out[f'{variant}_ct'], out[f'{variant}_tumor'], meta[variant] = crops_from(arrays)
        if variant == 'primary':
            flipped = {k: np.ascontiguousarray(v[:, :, ::-1]) for k, v in arrays.items()}
            out['S4_reversed_slices_ct'], out['S4_reversed_slices_tumor'], meta['S4_reversed_slices'] = crops_from(flipped)
    np.savez_compressed(target, **out)
    meta_file.write_text(json.dumps(meta))
    return meta

t0 = time.time()
CROP_META = {}
with ThreadPoolExecutor(N_IO) as pool:
    for n, (pid, meta) in enumerate(zip(INCLUDED, pool.map(preprocess, INCLUDED)), start=1):
        CROP_META[pid] = meta
        if n % 20 == 0 or n == len(INCLUDED):
            print(f'  preprocessed {n}/{len(INCLUDED)} ({(time.time() - t0) / 60:.1f} min)', flush=True)

crop_rows = [{'patient_id': pid, 'variant': v, **m} for pid, metas in CROP_META.items() for v, m in metas.items()]
pd.DataFrame(crop_rows).to_csv(OUT_DIR / 'E8_0_restored2_crop_geometry.csv', index=False)
(OUT_DIR / 'crops_primary_restored2').mkdir()
for pid in INCLUDED:
    with np.load(CROP_DIR / f'{pid}.npz') as z:
        np.savez_compressed(OUT_DIR / 'crops_primary_restored2' / f'{pid}.npz', ct=z['primary_ct'], tumor_mask=z['primary_tumor'])
geo = pd.DataFrame(crop_rows)
print('Tumour fraction inside the 96^3 crop (median by variant):', geo.groupby('variant')['tumor_in_crop'].median().round(3).to_dict())

In [ ]:
# Cell 5 — Frozen CNNs on every variant
def load_variant(variant):
    arrays = {}
    for pid in INCLUDED:
        with np.load(CROP_DIR / f'{pid}.npz') as z:
            arrays[pid] = z[f'{variant}_ct']
    return arrays

PROBS = {}                                    # (model, variant) -> DataFrame indexed by patient
t0 = time.time()
for variant in VARIANTS:
    arrays = load_variant(variant)
    for name, (models, channels) in CNNS.items():
        PROBS[(name, variant)] = cnn_probs(models, channels, arrays).loc[INCLUDED]
    print(f'  CNNs done for {variant} ({(time.time() - t0) / 60:.1f} min)', flush=True)

In [ ]:
# Cell 6 — Radiomics without shape features (E2 models) on every variant (Week 5 extraction: 0-255 crops, unit spacing)
def make_extractor():
    extractor = featureextractor.RadiomicsFeatureExtractor(binWidth=5)
    extractor.disableAllImageTypes()
    extractor.enableImageTypeByName('Original')
    extractor.disableAllFeatures()
    for feature_class in ('shape', 'firstorder', 'glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm'):
        extractor.enableFeatureClassByName(feature_class)
    return extractor

def extract_patient(pid, variant, crop_dir):
    extractor = make_extractor()
    with np.load(crop_dir / f'{pid}.npz') as z:
        ct4, tumor = z[f'{variant}_ct'], z[f'{variant}_tumor']
    features = {}
    if tumor.sum() == 0:
        return pid, features
    mask_img = sitk.GetImageFromArray(tumor.astype(np.uint8))
    for i, phase in enumerate(PHASE_ORDER):
        try:
            for key, value in extractor.execute(sitk.GetImageFromArray(ct4[i].astype(np.float32)), mask_img).items():
                if not key.startswith('diagnostics'):
                    try:
                        features[f'{phase}_{key}'] = float(value)
                    except (TypeError, ValueError):
                        pass
        except Exception as error:
            print(f'  radiomics failed for {pid}/{phase}: {error}')
    return pid, features

RAD = {}
t0 = time.time()
for variant in VARIANTS:
    jobs = [(pid, variant, CROP_DIR) for pid in INCLUDED]
    results = None
    if N_JOBS > 1:
        try:
            results = joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(extract_patient)(*job) for job in jobs)
        except Exception as error:                       # same results, only slower
            print(f'  parallel radiomics failed ({error!r:.120}); continuing one patient at a time')
    if results is None:
        results = [extract_patient(*job) for job in jobs]
    RAD[variant] = pd.DataFrame.from_dict(dict(results), orient='index').reindex(INCLUDED)
    RAD[variant].to_csv(OUT_DIR / f'E8_0_restored2_radiomics_features_{variant}.csv')
    print(f'  radiomics done for {variant}: {RAD[variant].shape} ({(time.time() - t0) / 60:.1f} min)', flush=True)

def radiomics_probs(features):
    per_fold, imputed, total = [], 0, 0
    for f, m in enumerate(rad_models):
        names = m['sel_names']
        train_ids = [p for p in dev_ids if p not in set(fold_splits[f]['val_ids'])]
        medians = internal.set_index('patient_id').loc[train_ids, names].median(axis=0, skipna=True)
        X = features.reindex(columns=names).to_numpy(dtype=np.float64)
        missing = ~np.isfinite(X)
        imputed += int(missing.sum())
        total += X.size
        X[missing] = np.broadcast_to(medians.to_numpy(), X.shape)[missing]
        per_fold.append(m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X.astype(np.float32)))))
    return pd.DataFrame(np.mean(per_fold, axis=0), index=features.index, columns=['p_HCC', 'p_ICC', 'p_CHCC']), imputed / total

IMPUTED = {}
for variant in VARIANTS:
    PROBS[('radiomics_noshape', variant)], IMPUTED[variant] = radiomics_probs(RAD[variant])
print('Radiomics cells filled with training medians (should be ~0):', {k: round(v, 4) for k, v in IMPUTED.items()})

In [ ]:
# Cell 7 — Fusion exactly as fitted internally (E2/E3/E5), with the real MCT-LTDiag age and sex
W4 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC']
W5 = ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
FUSIONS = {'cnn_plus_radiomics': W4 + W5, 'full_fusion': W4 + W5 + ['age', 'gender']}

oof_original = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str})
d_pred = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
d_oof = d_pred.query("pattern == 'complete' and split == 'oof'").set_index('patient_id')[['p_HCC', 'p_ICC', 'p_CHCC']]
oof_dropout = oof_original.copy()
oof_dropout[W4] = d_oof.loc[oof_dropout['patient_id']].to_numpy()
OOF_BY_CNN = {'original_CNN': oof_original, 'phase_dropout_CNN': oof_dropout}

def prepare(X):
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X

def fit_meta_folds(columns, oof):
    fitted = []
    for f in range(5):
        train = oof['fold'].astype(int).ne(f)
        X = prepare(oof.loc[train, columns].to_numpy())
        imputer = SimpleImputer(strategy='median').fit(X)
        scaler = StandardScaler().fit(imputer.transform(X))
        model = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED)
        fitted.append((imputer, scaler, model.fit(scaler.transform(imputer.transform(X)), oof.loc[train, 'label'].astype(int))))
    return fitted

def parse_age(value):
    match = re.fullmatch(r'0*(\d+)\s*[Yy]?', str(value).strip())
    return float(match.group(1)) if match else np.nan

clin = SEL.set_index('patient_id').loc[INCLUDED]
AGE = clin['patient_age'].map(parse_age).to_numpy()
SEX = clin['patient_sex'].str.strip().str.upper().map({'M': 1.0, 'F': 0.0}).to_numpy()   # internal coding
print(f'Clinical: age observed {np.isfinite(AGE).sum()}/{len(INCLUDED)} (median {np.nanmedian(AGE):.0f}), '
      f'sex observed {np.isfinite(SEX).sum()} (male {int((SEX == 1).sum())}, female {int((SEX == 0).sum())})')

FITTED = {(cnn, fusion): fit_meta_folds(cols, oof) for cnn, oof in OOF_BY_CNN.items() for fusion, cols in FUSIONS.items()}
for variant in VARIANTS:
    for (cnn_name, fusion_name), fitted in FITTED.items():
        frame = pd.DataFrame(index=INCLUDED)
        frame[W4] = PROBS[(cnn_name, variant)].loc[INCLUDED].to_numpy()
        frame[W5] = PROBS[('radiomics_noshape', variant)].loc[INCLUDED].to_numpy()
        frame['age'], frame['gender'] = AGE, SEX
        X = prepare(frame[FUSIONS[fusion_name]].to_numpy())
        probs = np.mean([model.predict_proba(scaler.transform(imputer.transform(X))) for imputer, scaler, model in fitted], axis=0)
        PROBS[(f'{fusion_name} [{cnn_name}]', variant)] = pd.DataFrame(probs, index=INCLUDED, columns=['p_HCC', 'p_ICC', 'p_CHCC'])
print('Models x variants:', len(PROBS))

In [ ]:
# Cell 8 — E8.0: merge the 2 restored patients with the E6 run (201) -> all 203; consistency check
RESTORED = list(INCLUDED)
ALL203 = SEL['patient_id'].tolist()
e6_pred = pd.read_csv(INPUTS['e6_predictions'][0], dtype={'patient_id': str})
PROBS_RESTORED = dict(PROBS)
PROBS203 = {}
for (model, variant), probs in PROBS_RESTORED.items():
    old = e6_pred[(e6_pred['model'] == model) & (e6_pred['variant'] == variant)].set_index('patient_id')[['p_HCC', 'p_ICC', 'p_CHCC']]
    merged = pd.concat([old, probs.loc[RESTORED]])
    if merged.index.duplicated().any() or set(merged.index) != set(ALL203):
        raise RuntimeError(f'STOP — merging failed for {model} / {variant}.')
    PROBS203[(model, variant)] = merged.loc[ALL203]
RAD203 = {}
for variant in VARIANTS:
    old = pd.read_csv(INPUTS[f'e6_rad_{variant}'][0], index_col=0)
    old.index = old.index.astype(str)
    RAD203[variant] = pd.concat([old, RAD[variant].loc[RESTORED]]).reindex(ALL203)
INCLUDED = ALL203
E80_CHECK = {}
for variant in VARIANTS:
    p203, IMPUTED[variant] = radiomics_probs(RAD203[variant])
    old_ids = [p for p in ALL203 if p not in RESTORED]
    E80_CHECK[variant] = float(np.abs(p203.loc[old_ids].to_numpy() - PROBS203[('radiomics_noshape', variant)].loc[old_ids].to_numpy()).max())
print('E8.0 consistency (frozen radiomics on the merged features vs E6 saved probabilities), max difference:', E80_CHECK)
if IN_COLAB and max(E80_CHECK.values()) > 1e-6:
    raise RuntimeError('STOP — merged data do not reproduce the E6 run. Send this output.')
PROBS = PROBS203
TECH_FAIL = TECH_FAIL.iloc[0:0]
print(f'E8.0: cohort restored to {len(INCLUDED)} patients (restored: {RESTORED})')

In [ ]:
# Cell 9 — E8.0 results: E6's results code on all 203 patients
PRIMARY = ('cnn_plus_radiomics [phase_dropout_CNN]', 'primary')
MODEL_ORDER = ['cnn_plus_radiomics [phase_dropout_CNN]', 'full_fusion [phase_dropout_CNN]', 'phase_dropout_CNN',
               'radiomics_noshape', 'original_CNN', 'cnn_plus_radiomics [original_CNN]', 'full_fusion [original_CNN]',
               'single_phase_CNN']
N_BOOT = 2000
TRUE = SEL.set_index('patient_id').loc[INCLUDED, 'type']
Y = (TRUE == 'HCC').astype(int).to_numpy()                   # HCC = positive class
PRIOR = SEL.set_index('patient_id').loc[INCLUDED, 'History of chemotherapy (Y/N)'].fillna('').str.strip().str.upper().str.startswith('Y').to_numpy()
SUBSETS = {'all': np.ones(len(INCLUDED), bool), 'no_prior_treatment': ~PRIOR}

def exact_ci(k, n):
    low = 0.0 if k == 0 else float(beta.ppf(0.025, k, n - k + 1))
    high = 1.0 if k == n else float(beta.ppf(0.975, k + 1, n - k))
    return low, high

def compute_midrank(values):
    order = np.argsort(values)
    sorted_values = values[order]
    n_values = len(values)
    midranks = np.zeros(n_values, dtype=np.float64)
    index = 0
    while index < n_values:
        end = index
        while end < n_values and sorted_values[end] == sorted_values[index]:
            end += 1
        midranks[index:end] = 0.5 * (index + end - 1) + 1
        index = end
    output = np.empty(n_values, dtype=np.float64)
    output[order] = midranks
    return output

def delong_components(y, scores):
    pos, neg = scores[y == 1], scores[y == 0]
    m, n = len(pos), len(neg)
    tx, ty, tz = compute_midrank(pos), compute_midrank(neg), compute_midrank(np.concatenate([pos, neg]))
    auc = tz[:m].sum() / m / n - (m + 1.0) / 2.0 / n
    return auc, (tz[:m] - tx) / n, 1.0 - (tz[m:] - ty) / m

def delong_ci(y, s):
    auc, v01, v10 = delong_components(y, s)
    se = np.sqrt(np.var(v01, ddof=1) / len(v01) + np.var(v10, ddof=1) / len(v10))
    return float(auc), float(max(0.0, auc - 1.96 * se)), float(min(1.0, auc + 1.96 * se))

def delong_paired_p(y, s1, s2):
    a1, v01a, v10a = delong_components(y, s1)
    a2, v01b, v10b = delong_components(y, s2)
    cov = np.cov(np.vstack([v01a, v01b])) / len(v01a) + np.cov(np.vstack([v10a, v10b])) / len(v10a)
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    return 1.0 if var <= 0 else float(2 * norm.sf(abs(a1 - a2) / np.sqrt(var)))

rng = np.random.default_rng(SEED)
BOOT = [rng.integers(0, len(INCLUDED), len(INCLUDED)) for _ in range(N_BOOT)]

def score(probs):
    p = probs.loc[INCLUDED].to_numpy()
    return p[:, 0] / np.clip(p[:, 0] + p[:, 1], 1e-12, None), p.argmax(axis=1)

def boot_auc(y, s):
    vals = [roc_auc_score(y[b], s[b]) for b in BOOT if 0 < y[b].sum() < len(b)]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

rows = []
for (model, variant), probs in PROBS.items():
    s_all, pred_all = score(probs)
    for subset, keep in SUBSETS.items():
        if subset != 'all' and variant != 'primary':
            continue
        y, s, pred = Y[keep], s_all[keep], pred_all[keep]
        auc, d_low, d_high = delong_ci(y, s)
        b_low, b_high = boot_auc(y, s) if subset == 'all' else (np.nan, np.nan)
        k_sens, n_hcc = int(((pred == 0) & (y == 1)).sum()), int((y == 1).sum())
        k_spec, n_icc = int(((pred == 1) & (y == 0)).sum()), int((y == 0).sum())
        rows.append({'model': model, 'variant': variant, 'subset': subset, 'n': int(len(y)), 'n_HCC': n_hcc, 'n_ICC': n_icc,
                     'auc': auc, 'auc_delong_low': d_low, 'auc_delong_high': d_high, 'auc_boot_low': b_low, 'auc_boot_high': b_high,
                     'hcc_sensitivity': k_sens / n_hcc, 'sens_ci_low': exact_ci(k_sens, n_hcc)[0], 'sens_ci_high': exact_ci(k_sens, n_hcc)[1],
                     'icc_specificity': k_spec / n_icc, 'spec_ci_low': exact_ci(k_spec, n_icc)[0], 'spec_ci_high': exact_ci(k_spec, n_icc)[1],
                     **{f'HCC_pred_{c}': int(((pred == i) & (y == 1)).sum()) for i, c in enumerate(CLASSES)},
                     **{f'ICC_pred_{c}': int(((pred == i) & (y == 0)).sum()) for i, c in enumerate(CLASSES)}})
RESULTS = pd.DataFrame(rows)
RESULTS.to_csv(OUT_DIR / 'E8_0_E6_203_results.csv', index=False)

PAIRS = [('phase_dropout_CNN', 'original_CNN'), ('full_fusion [phase_dropout_CNN]', 'cnn_plus_radiomics [phase_dropout_CNN]'),
         ('radiomics_noshape', 'phase_dropout_CNN'), ('cnn_plus_radiomics [phase_dropout_CNN]', 'radiomics_noshape')]
pair_rows = []
for a, b in PAIRS:
    sa, _ = score(PROBS[(a, 'primary')]); sb, _ = score(PROBS[(b, 'primary')])
    diffs = [roc_auc_score(Y[i], sa[i]) - roc_auc_score(Y[i], sb[i]) for i in BOOT if 0 < Y[i].sum() < len(i)]
    pair_rows.append({'model_a': a, 'model_b': b, 'auc_a': roc_auc_score(Y, sa), 'auc_b': roc_auc_score(Y, sb),
                      'difference': roc_auc_score(Y, sa) - roc_auc_score(Y, sb), 'boot_low': float(np.percentile(diffs, 2.5)),
                      'boot_high': float(np.percentile(diffs, 97.5)), 'delong_p': delong_paired_p(Y, sa, sb)})
PAIRED = pd.DataFrame(pair_rows)
PAIRED.to_csv(OUT_DIR / 'E8_0_E6_203_paired_differences.csv', index=False)

long = []
for (model, variant), probs in PROBS.items():
    t = probs.loc[INCLUDED].copy()
    t.insert(0, 'patient_id', t.index)
    t.insert(0, 'variant', variant)
    t.insert(0, 'model', model)
    t['true_class'] = TRUE.to_numpy()
    t['prior_treatment'] = PRIOR
    long.append(t)
pd.concat(long, ignore_index=True).to_csv(OUT_DIR / 'E8_0_E6_203_predictions.csv', index=False)

prim = RESULTS[(RESULTS['model'] == PRIMARY[0]) & (RESULTS['variant'] == PRIMARY[1]) & (RESULTS['subset'] == 'all')].iloc[0]
summary = {'run_id': RUN_ID, 'notebook': 'E8a_MCT203_and_radiomics_mitigation.ipynb (E8.0 with E6 code)', 'versions': VERSIONS,
           'input_sha256': input_hashes, 'internal_reproduction_check': gate, 'internal_orientation_reference': REF,
           'included': len(INCLUDED), 'technical_failures': TECH_FAIL.to_dict('records'), 'facts': FACTS_SUMMARY,
           'orientation': orientation_summary, 'radiomics_imputed_fraction': IMPUTED,
           'primary': {'model': PRIMARY[0], 'variant': PRIMARY[1], 'auc': prim['auc'],
                       'delong_ci': [prim['auc_delong_low'], prim['auc_delong_high']],
                       'bootstrap_ci': [prim['auc_boot_low'], prim['auc_boot_high']],
                       'hcc_sensitivity': prim['hcc_sensitivity'], 'icc_specificity': prim['icc_specificity']},
           'protocol': '00 log section 17 (approved 17 Oct 2026) with amendments 17.9; no tuning on MCT-LTDiag; argmax decisions'}
(OUT_DIR / 'E8_0_summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest_out = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)}
                for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest_out).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

def fmt(r):
    return (f"AUC {r['auc']:.3f} ({r['auc_delong_low']:.3f}-{r['auc_delong_high']:.3f}) | "
            f"HCC sens {r['hcc_sensitivity']:.2f} ({int(r['HCC_pred_HCC'])}/{int(r['n_HCC'])}) | "
            f"ICC spec {r['icc_specificity']:.2f} ({int(r['ICC_pred_ICC'])}/{int(r['n_ICC'])}) | "
            f"HCC->[H,I,C] {int(r['HCC_pred_HCC'])},{int(r['HCC_pred_ICC'])},{int(r['HCC_pred_CHCC'])} "
            f"ICC->[H,I,C] {int(r['ICC_pred_HCC'])},{int(r['ICC_pred_ICC'])},{int(r['ICC_pred_CHCC'])}")

print('\n---------------- E8.0: E6 on all 203 patients ----------------')
print(f'E8.0 in run E8a_{RUN_ID} | internal check passed: {GATE_OK} | included {len(INCLUDED)} '
      f'(HCC {int(Y.sum())}, ICC {int((1 - Y).sum())}) | technical failures {len(TECH_FAIL)}')
print('Facts:', json.dumps(FACTS_SUMMARY, default=str))
print('Orientation (2 restored patients):', orientation_summary,
      '| internal ref medians:', {k: round(REF[k], 2) for k in ['median_liver_i', 'median_bone_j', 'median_bulk_k']})
print(f"PRIMARY  {PRIMARY[0]}, primary preprocessing: AUC {prim['auc']:.3f} (DeLong {prim['auc_delong_low']:.3f}-"
      f"{prim['auc_delong_high']:.3f}; bootstrap {prim['auc_boot_low']:.3f}-{prim['auc_boot_high']:.3f})")
print('\n--- all models, primary preprocessing, all patients ---')
for model in MODEL_ORDER:
    r = RESULTS[(RESULTS['model'] == model) & (RESULTS['variant'] == 'primary') & (RESULTS['subset'] == 'all')].iloc[0]
    print(f'{model:42s} {fmt(r)}')
print('\n--- AUC by preprocessing variant (sensitivity analyses) and without prior treatment ---')
table = RESULTS[RESULTS['subset'] == 'all'].pivot(index='model', columns='variant', values='auc').reindex(MODEL_ORDER)[VARIANTS]
table['S3_no_prior_treatment'] = RESULTS[RESULTS['subset'] == 'no_prior_treatment'].set_index('model')['auc'].reindex(MODEL_ORDER)
print(table.round(3).to_string())
print('\n--- paired AUC differences, primary preprocessing (bootstrap 95% CI, DeLong p) ---')
print(PAIRED.round(3).to_string(index=False))
print('\nRadiomics median-filled fraction:', {k: round(v, 4) for k, v in IMPUTED.items()})
print('Versions:', VERSIONS)
print('---------------------------------------------------------------')

RESULTS_E80, PAIRED_E80, PRIM_E80 = RESULTS.copy(), PAIRED.copy(), prim.copy()

In [ ]:
# Cell 10 — E8a.1: reference-batch ComBat (internal development = reference; MCT-LTDiag mapped; no labels, no covariates)

for _alias, _builtin in {'int': int, 'float': float, 'bool': bool, 'object': object}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _builtin)   # neuroCombat still uses np.int (removed in NumPy 1.24); identical behaviour

POOL = rad_models[0]['feature_pool']
ref = internal.set_index('patient_id').loc[dev_ids, POOL].astype(float)
ext = RAD203['primary'].reindex(columns=POOL).astype(float)
fill = ref.median()
A, B = ref.fillna(fill), ext.fillna(fill)
USABLE = [c for c in POOL if A[c].std() > 1e-8 and B[c].std() > 1e-8]
combat = neuroCombat(dat=pd.concat([A[USABLE], B[USABLE]]).T.to_numpy(),
                     covars=pd.DataFrame({'batch': ['internal'] * len(A) + ['MCT'] * len(B)}),
                     batch_col='batch', ref_batch='internal')
adjusted = pd.DataFrame(np.asarray(combat['data']).T, index=list(A.index) + list(B.index), columns=USABLE)
REF_CHANGE = float(np.abs(adjusted.loc[A.index].to_numpy() - A[USABLE].to_numpy()).max() / (np.abs(A[USABLE].to_numpy()).max() + 1e-12))
MCT_COMBAT = ext.copy()
MCT_COMBAT[USABLE] = adjusted.loc[B.index, USABLE].to_numpy()
print(f'ComBat: {len(USABLE)}/{len(POOL)} features harmonised; reference batch relative change {REF_CHANGE:.1e} (should be ~0)')
if IN_COLAB and REF_CHANGE > 1e-6:
    raise RuntimeError('STOP — ComBat changed the internal (reference) features. Send this output.')
MIT = {}                                       # mitigation probabilities, MCT-LTDiag 203, primary preprocessing
MIT['radiomics_noshape'] = PROBS[('radiomics_noshape', 'primary')]
MIT['radiomics_noshape_ComBat'], _ = radiomics_probs(MCT_COMBAT)

In [ ]:
# Cell 11 — E8a.2: perturbation-robust features (internal crops only), ICC(1,1) >= 0.90, E2 recipe retrained on them
PERTURBATIONS = ['baseline', 'shift', 'rotate', 'noise', 'erode', 'dilate']
ICC_THRESHOLD = 0.90

def perturbed_features(pid, processed_dir, seed):
    extractor = make_extractor()
    with np.load(processed_dir / f'{pid}.npz', allow_pickle=False) as z:
        ct4, tm4 = z['ct'].astype(np.float32), z['tumor_mask'].astype(np.uint8)
    rng = np.random.default_rng(seed)
    out = {}
    for kind in PERTURBATIONS:
        feats = {}
        for i, phase in enumerate(PHASE_ORDER):
            ct, m = ct4[i], tm4[i] > 0
            if kind == 'shift':
                ct = ndimage.shift(ct, (1, 1, 1), order=1, mode='constant')
                m = ndimage.shift(m.astype(np.uint8), (1, 1, 1), order=0, mode='constant') > 0
            elif kind == 'rotate':
                ct = ndimage.rotate(ct, 5, axes=(0, 1), reshape=False, order=1, mode='constant')
                m = ndimage.rotate(m.astype(np.uint8), 5, axes=(0, 1), reshape=False, order=0, mode='constant') > 0
            elif kind == 'noise':
                ct = np.where(ct > 0, np.clip(ct + rng.normal(0, 2, ct.shape), 0, 255), 0).astype(np.float32)
            elif kind == 'erode':
                m = ndimage.binary_erosion(m)
            elif kind == 'dilate':
                m = ndimage.binary_dilation(m)
            if m.sum() < 10:
                continue
            try:
                result = extractor.execute(sitk.GetImageFromArray(ct.astype(np.float32)), sitk.GetImageFromArray(m.astype(np.uint8)))
            except Exception:
                continue
            for key, value in result.items():
                if not key.startswith('diagnostics'):
                    try:
                        feats[f'{phase}_{key}'] = float(value)
                    except (TypeError, ValueError):
                        pass
        out[kind] = feats
    return pid, out

all_ids = dev_ids + test_ids
t0 = time.time()
jobs = [(pid, PROCESSED_DIR, SEED + n) for n, pid in enumerate(all_ids)]
pert = None
if N_JOBS > 1:
    try:
        pert = joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(perturbed_features)(*job) for job in jobs)
    except Exception as error:
        print(f'  parallel extraction failed ({error!r:.120}); continuing one patient at a time')
if pert is None:
    pert = [perturbed_features(*job) for job in jobs]
long_rows = [{'patient_id': pid, 'perturbation': kind, **feats} for pid, versions in pert for kind, feats in versions.items()]
PERT = pd.DataFrame(long_rows)
PERT.to_csv(OUT_DIR / 'E8a_perturbation_features.csv.gz', index=False, compression='gzip')
print(f'Perturbation features: {PERT.shape} ({(time.time() - t0) / 60:.1f} min)')

def icc11(M):
    n, k = M.shape
    grand, rows = M.mean(), M.mean(axis=1)
    msb = k * ((rows - grand) ** 2).sum() / (n - 1)
    msw = ((M - rows[:, None]) ** 2).sum() / (n * (k - 1))
    return float((msb - msw) / (msb + (k - 1) * msw)) if (msb + (k - 1) * msw) > 0 else np.nan

ICC = {}
for c in POOL:
    if c not in PERT.columns:
        ICC[c] = np.nan
        continue
    wide = PERT.pivot(index='patient_id', columns='perturbation', values=c).reindex(columns=PERTURBATIONS).dropna()
    ICC[c] = icc11(wide.to_numpy(dtype=float)) if len(wide) >= 20 else np.nan
ICC_TABLE = pd.DataFrame({'feature': list(ICC), 'icc': list(ICC.values())})
ICC_TABLE['phase'] = ICC_TABLE['feature'].str.split('_').str[0]
ICC_TABLE['robust'] = ICC_TABLE['icc'] >= ICC_THRESHOLD
ICC_TABLE.to_csv(OUT_DIR / 'E8a_icc_per_feature.csv', index=False)
ROBUST = ICC_TABLE.loc[ICC_TABLE['robust'], 'feature'].tolist()
(OUT_DIR / 'E8a_robust_features.json').write_text(json.dumps(ROBUST, indent=1))
e2_selected = sorted({n for m in rad_models for n in m['sel_names']})
print(f'Robust features: {len(ROBUST)}/{len(POOL)} (ICC >= {ICC_THRESHOLD}); per phase', ICC_TABLE.groupby('phase')['robust'].sum().to_dict(),
      f'| E2-selected features that are robust: {sum(n in ROBUST for n in e2_selected)}/{len(e2_selected)}')
if len(ROBUST) < 5:
    raise RuntimeError('STOP — fewer than 5 robust features. Send this output.')

# ---- E2 recipe (Week 5), unchanged, on the robust features ----
K_TOP_FEATURES, CORR_THRESHOLD = 30, 0.95

def select_features_train(X_tr, y_tr, feature_names, k=K_TOP_FEATURES, corr_thresh=CORR_THRESHOLD):
    stds = np.nanstd(X_tr, axis=0)
    keep = stds > 1e-10
    X_kept = X_tr[:, keep]
    kept_idx_global = np.where(keep)[0]
    X_nan_to_num = np.nan_to_num(X_kept, nan=np.nanmedian(X_kept))
    corr = np.abs(np.corrcoef(X_nan_to_num.T))
    remove = np.zeros(corr.shape[0], dtype=bool)
    for i in range(corr.shape[0]):
        if remove[i]:
            continue
        for j in range(i + 1, corr.shape[0]):
            if remove[j]:
                continue
            if corr[i, j] > corr_thresh:
                remove[j] = True
    kept_idx_after_corr = kept_idx_global[~remove]
    X_for_f = SimpleImputer(strategy='median').fit_transform(X_tr[:, kept_idx_after_corr])
    F, _ = f_classif(X_for_f, y_tr)
    order = np.argsort(F)[::-1]
    final_idx = kept_idx_after_corr[order[:min(k, len(order))]]
    return final_idx, [feature_names[i] for i in final_idx]

def train_one_fold_radiomics(fold_idx, train_ids, val_ids, df_features, feature_cols):
    df_tr = df_features[df_features['patient_id'].isin(train_ids)].reset_index(drop=True)
    df_va = df_features[df_features['patient_id'].isin(val_ids)].reset_index(drop=True)
    X_tr_all = df_tr[feature_cols].values.astype(np.float32)
    y_tr = df_tr['label'].values
    X_va_all = df_va[feature_cols].values.astype(np.float32)
    y_va = df_va['label'].values
    sel_idx, sel_names = select_features_train(X_tr_all, y_tr, feature_cols)
    X_tr = X_tr_all[:, sel_idx]
    X_va = X_va_all[:, sel_idx]
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr)
    X_va = imp.transform(X_va)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_va = scaler.transform(X_va)
    clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, num_leaves=15,
                             min_child_samples=5, reg_alpha=0.1, reg_lambda=0.1,
                             class_weight='balanced', random_state=42, verbose=-1)
    clf.fit(X_tr, y_tr)
    probs = clf.predict_proba(X_va)
    return {'fold': fold_idx, 'auc': float(roc_auc_score(y_va, probs, multi_class='ovr', average='macro')),
            'probs': probs, 'labels': y_va, 'patient_ids': df_va['patient_id'].tolist(),
            'selected_features': sel_names, 'sel_idx': sel_idx, 'imputer': imp, 'scaler': scaler, 'clf': clf}

def run_radiomics(df_features, feature_cols, fold_splits, test_ids):
    fold_results = [train_one_fold_radiomics(f, s['train_ids'], s['val_ids'], df_features, feature_cols)
                    for f, s in enumerate(fold_splits)]
    oof = pd.concat([pd.DataFrame({'patient_id': r['patient_ids'], 'fold': r['fold'], 'label': r['labels'],
                                   'p_HCC': r['probs'][:, 0], 'p_ICC': r['probs'][:, 1], 'p_CHCC': r['probs'][:, 2]})
                     for r in fold_results], ignore_index=True)
    df_test = df_features[df_features['patient_id'].isin(test_ids)].reset_index(drop=True)
    X_test_all = df_test[feature_cols].values.astype(np.float32)
    probs = np.mean([r['clf'].predict_proba(r['scaler'].transform(r['imputer'].transform(X_test_all[:, r['sel_idx']])))
                     for r in fold_results], axis=0)
    test = pd.DataFrame({'patient_id': df_test['patient_id'].values, 'label': df_test['label'].values,
                         'p_HCC': probs[:, 0], 'p_ICC': probs[:, 1], 'p_CHCC': probs[:, 2]})
    return fold_results, oof, test

def apply_fold_models(fold_results, features, feature_cols):
    X = features.reindex(columns=feature_cols).to_numpy(dtype=np.float32)
    probs = np.mean([r['clf'].predict_proba(r['scaler'].transform(r['imputer'].transform(X[:, r['sel_idx']])))
                     for r in fold_results], axis=0)
    return pd.DataFrame(probs, index=features.index, columns=['p_HCC', 'p_ICC', 'p_CHCC'])

ROBUST_FOLDS, ROBUST_OOF, ROBUST_TEST = run_radiomics(internal, ROBUST, fold_splits, test_ids)
MIT['radiomics_robust'] = apply_fold_models(ROBUST_FOLDS, RAD203['primary'], ROBUST)
MIT['radiomics_robust_ComBat'] = apply_fold_models(ROBUST_FOLDS, MCT_COMBAT, ROBUST)

In [ ]:
# Cell 12 — E8a fusions with the phase-dropout CNN (internal OOF meta-learners), internal test and HCC-TACE-Seg checks
clin203 = SEL.set_index('patient_id').loc[ALL203]
AGE203 = clin203['patient_age'].map(parse_age).to_numpy()
SEX203 = clin203['patient_sex'].str.strip().str.upper().map({'M': 1.0, 'F': 0.0}).to_numpy()
e2_oof_rad = pd.read_csv(INPUTS['e2_radiomics_oof'][0], dtype={'patient_id': str}).set_index('patient_id')
test_meta = pd.read_csv(INPUTS['e2_test_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids]
d_test = d_pred.query("pattern == 'complete' and split == 'test'").set_index('patient_id')[['p_HCC', 'p_ICC', 'p_CHCC']].loc[test_ids]
RAD_INTERNAL = {'radiomics_noshape': (e2_oof_rad[['p_HCC', 'p_ICC', 'p_CHCC']], test_meta[W5].set_axis(['p_HCC', 'p_ICC', 'p_CHCC'], axis=1)),
                'radiomics_robust': (ROBUST_OOF.set_index('patient_id')[['p_HCC', 'p_ICC', 'p_CHCC']],
                                     ROBUST_TEST.set_index('patient_id')[['p_HCC', 'p_ICC', 'p_CHCC']])}
RAD_INTERNAL['radiomics_noshape_ComBat'] = RAD_INTERNAL['radiomics_noshape']            # frozen models, internal unchanged
RAD_INTERNAL['radiomics_robust_ComBat'] = RAD_INTERNAL['radiomics_robust']
y_int = np.array([TEST_LABEL[p] for p in test_ids])
INTERNAL_ROWS = []
for rad_name, (oof_rad, test_rad) in RAD_INTERNAL.items():
    oof = oof_dropout.copy()
    oof[W5] = oof_rad.loc[oof['patient_id']].to_numpy()
    test_frame = pd.DataFrame(index=test_ids)
    test_frame[W4] = d_test.to_numpy()
    test_frame[W5] = test_rad.loc[test_ids].to_numpy()
    test_frame['age'], test_frame['gender'] = test_meta['age'].to_numpy(), test_meta['gender'].to_numpy()
    INTERNAL_ROWS.append({'model': rad_name, 'internal_test_macro_auc': macro_auc3(y_int, test_rad.loc[test_ids].to_numpy())})
    for fusion_name, columns in FUSIONS.items():
        fitted = fit_meta_folds(columns, oof)
        label = f'{fusion_name} [phase_dropout_CNN] + {rad_name}'
        p_test = np.mean([m.predict_proba(s.transform(i.transform(prepare(test_frame[columns].to_numpy())))) for i, s, m in fitted], axis=0)
        INTERNAL_ROWS.append({'model': label, 'internal_test_macro_auc': macro_auc3(y_int, p_test)})
        frame = pd.DataFrame(index=ALL203)
        frame[W4] = PROBS[('phase_dropout_CNN', 'primary')].loc[ALL203].to_numpy()
        frame[W5] = MIT[rad_name].loc[ALL203].to_numpy()
        frame['age'], frame['gender'] = AGE203, SEX203
        MIT[label] = pd.DataFrame(np.mean([m.predict_proba(s.transform(i.transform(prepare(frame[columns].to_numpy()))))
                                           for i, s, m in fitted], axis=0), index=ALL203, columns=['p_HCC', 'p_ICC', 'p_CHCC'])
INTERNAL = pd.DataFrame(INTERNAL_ROWS)
cv_e2 = [roc_auc_score(g['label'], g[['p_HCC', 'p_ICC', 'p_CHCC']], multi_class='ovr', average='macro')
         for _, g in e2_oof_rad.reset_index().groupby('fold')]
cv_robust = [r['auc'] for r in ROBUST_FOLDS]
print('Internal held-out macro-AUC:\n' + INTERNAL.round(3).to_string(index=False))
print(f'Internal CV macro-AUC: E2 radiomics {np.mean(cv_e2):.3f} ± {np.std(cv_e2):.3f} | robust radiomics {np.mean(cv_robust):.3f} ± {np.std(cv_robust):.3f}')
ref_rows = INTERNAL.set_index('model')['internal_test_macro_auc']
CONSISTENCY = {'cnn_plus_radiomics_D_E2_vs_E3': float(ref_rows['cnn_plus_radiomics [phase_dropout_CNN] + radiomics_noshape']),
               'full_fusion_D_E2_vs_E3': float(ref_rows['full_fusion [phase_dropout_CNN] + radiomics_noshape'])}
print('Consistency with E3 (expected 0.952 and 0.951):', {k: round(v, 3) for k, v in CONSISTENCY.items()})
if IN_COLAB and (abs(CONSISTENCY['cnn_plus_radiomics_D_E2_vs_E3'] - 0.952) > 0.002 or abs(CONSISTENCY['full_fusion_D_E2_vs_E3'] - 0.951) > 0.002):
    raise RuntimeError('STOP — the internal fusion does not reproduce E3. Send this output.')

# ---- HCC-TACE-Seg corrected (E5c primary corrected geometry): robust radiomics and its fusions ----
e5c = pd.read_csv(INPUTS['e5c_predictions'][0], dtype={'patient_id': str})
e5c = e5c[e5c['variant'] == 'axis_fixed_E6rules']
e5c_rad = pd.read_csv(INPUTS['e5c_rad_E6rules'][0], index_col=0)
e5c_rad.index = e5c_rad.index.astype(str)
TACE_IDS = e5c_rad.index.tolist()
mixed = set(pd.read_csv(INPUTS['e5c_mixed'][0], dtype={'patient_id': str})['patient_id'])
TACE_CLEAN = [p for p in TACE_IDS if p not in mixed]
tace_clin = pd.read_csv(INPUTS['e5c_clinical'][0], dtype={'patient_id': str}).set_index('patient_id').reindex(TACE_IDS)
p_e2_tace, _ = radiomics_probs(e5c_rad)
saved_e2_tace = e5c[e5c['model'] == 'radiomics_noshape'].set_index('patient_id').loc[TACE_IDS, ['p_HCC', 'p_ICC', 'p_CHCC']]
TACE_CHECK = float(np.abs(p_e2_tace.loc[TACE_IDS].to_numpy() - saved_e2_tace.to_numpy()).max())
TACE = {'radiomics_noshape': p_e2_tace, 'radiomics_robust': apply_fold_models(ROBUST_FOLDS, e5c_rad, ROBUST)}
d_tace = e5c[e5c['model'] == 'phase_dropout_CNN'].set_index('patient_id').loc[TACE_IDS, ['p_HCC', 'p_ICC', 'p_CHCC']]
for rad_name in ['radiomics_noshape', 'radiomics_robust']:
    oof = oof_dropout.copy()
    oof[W5] = RAD_INTERNAL[rad_name][0].loc[oof['patient_id']].to_numpy()
    for fusion_name, columns in FUSIONS.items():
        fitted = fit_meta_folds(columns, oof)
        frame = pd.DataFrame(index=TACE_IDS)
        frame[W4], frame[W5] = d_tace.to_numpy(), TACE[rad_name].loc[TACE_IDS].to_numpy()
        frame['age'], frame['gender'] = tace_clin['age'].to_numpy(), tace_clin['sex'].to_numpy()
        TACE[f'{fusion_name} [phase_dropout_CNN] + {rad_name}'] = pd.DataFrame(
            np.mean([m.predict_proba(s.transform(i.transform(prepare(frame[columns].to_numpy())))) for i, s, m in fitted], axis=0),
            index=TACE_IDS, columns=['p_HCC', 'p_ICC', 'p_CHCC'])
saved_fusion_tace = e5c[e5c['model'] == 'cnn_plus_radiomics [phase_dropout_CNN]'].set_index('patient_id').loc[TACE_IDS, ['p_HCC', 'p_ICC', 'p_CHCC']]
TACE_CHECK = max(TACE_CHECK, float(np.abs(TACE['cnn_plus_radiomics [phase_dropout_CNN] + radiomics_noshape'].loc[TACE_IDS].to_numpy()
                                         - saved_fusion_tace.to_numpy()).max()))
print(f'HCC-TACE-Seg check (E2 radiomics and CNN + radiomics vs E5c saved): {TACE_CHECK:.1e}')
if IN_COLAB and TACE_CHECK > 1e-6:
    raise RuntimeError('STOP — this notebook does not reproduce E5c on HCC-TACE-Seg. Send this output.')

In [ ]:
# Cell 13 — E8a results (MCT-LTDiag 203: AUC with DeLong and bootstrap CIs; paired differences), saved outputs
MIT_ORDER = ['radiomics_noshape', 'radiomics_noshape_ComBat', 'radiomics_robust', 'radiomics_robust_ComBat']
FUSION_ORDER = [f'{f} [phase_dropout_CNN] + {r}' for f in ['cnn_plus_radiomics', 'full_fusion'] for r in MIT_ORDER]
rows = []
for model in MIT_ORDER + FUSION_ORDER:
    s_all, pred = score(MIT[model])
    auc, d_low, d_high = delong_ci(Y, s_all)
    b_low, b_high = boot_auc(Y, s_all)
    k_sens, n_hcc = int(((pred == 0) & (Y == 1)).sum()), int((Y == 1).sum())
    k_spec, n_icc = int(((pred == 1) & (Y == 0)).sum()), int((Y == 0).sum())
    rows.append({'model': model, 'n': len(Y), 'auc': auc, 'auc_delong_low': d_low, 'auc_delong_high': d_high,
                 'auc_boot_low': b_low, 'auc_boot_high': b_high, 'hcc_sensitivity': k_sens / n_hcc, 'icc_specificity': k_spec / n_icc,
                 **{f'HCC_pred_{c}': int(((pred == i) & (Y == 1)).sum()) for i, c in enumerate(CLASSES)},
                 **{f'ICC_pred_{c}': int(((pred == i) & (Y == 0)).sum()) for i, c in enumerate(CLASSES)}})
MIT_RESULTS = pd.DataFrame(rows)
MIT_RESULTS.to_csv(OUT_DIR / 'E8a_mct203_results.csv', index=False)

pairs = [(m, 'radiomics_noshape') for m in MIT_ORDER[1:]] + \
        [(f'{f} [phase_dropout_CNN] + {r}', f'{f} [phase_dropout_CNN] + radiomics_noshape') for f in ['cnn_plus_radiomics', 'full_fusion'] for r in MIT_ORDER[1:]]
pair_rows = []
for a, b in pairs:
    sa, _ = score(MIT[a]); sb, _ = score(MIT[b])
    diffs = [roc_auc_score(Y[i], sa[i]) - roc_auc_score(Y[i], sb[i]) for i in BOOT if 0 < Y[i].sum() < len(i)]
    pair_rows.append({'mitigated': a, 'reference': b, 'auc_mitigated': roc_auc_score(Y, sa), 'auc_reference': roc_auc_score(Y, sb),
                      'difference': roc_auc_score(Y, sa) - roc_auc_score(Y, sb), 'boot_low': float(np.percentile(diffs, 2.5)),
                      'boot_high': float(np.percentile(diffs, 97.5)), 'delong_p': delong_paired_p(Y, sa, sb)})
MIT_PAIRED = pd.DataFrame(pair_rows)
MIT_PAIRED.to_csv(OUT_DIR / 'E8a_paired_differences.csv', index=False)

def exact_rate(k, n):
    low, high = exact_ci(k, n)
    return f'{k}/{n} = {k / n:.2f} ({low:.2f}-{high:.2f})'
tace_rows = []
for model, probs in TACE.items():
    for subset, ids in [('clean_89', TACE_CLEAN), ('all_103', TACE_IDS)]:
        pred = probs.loc[ids].to_numpy().argmax(axis=1)
        tace_rows.append({'model': model, 'subset': subset, 'hcc_correct': int((pred == 0).sum()), 'n': len(ids),
                          'hcc_sensitivity': float((pred == 0).mean()), 'pred_ICC': int((pred == 1).sum()), 'pred_CHCC': int((pred == 2).sum())})
TACE_RESULTS = pd.DataFrame(tace_rows)
TACE_RESULTS.to_csv(OUT_DIR / 'E8a_hcc_tace_seg_results.csv', index=False)
INTERNAL.to_csv(OUT_DIR / 'E8a_internal_test.csv', index=False)
pred_long = []
for model, probs in MIT.items():
    t = probs.loc[ALL203].copy(); t.insert(0, 'patient_id', t.index); t.insert(0, 'model', model); t['true_class'] = TRUE.to_numpy()
    pred_long.append(t)
pd.concat(pred_long, ignore_index=True).to_csv(OUT_DIR / 'E8a_mct203_predictions.csv', index=False)

summary = {'run_id': RUN_ID, 'notebook': 'E8a_MCT203_and_radiomics_mitigation.ipynb', 'versions': VERSIONS,
           'input_sha256': input_hashes, 'internal_reproduction_check': gate, 'E8_0': {
               'restored': RESTORED, 'consistency_max_diff': E80_CHECK, 'orientation': orientation_summary,
               'primary_auc_203': float(PRIM_E80['auc']), 'primary_delong_ci': [float(PRIM_E80['auc_delong_low']), float(PRIM_E80['auc_delong_high'])]},
           'combat': {'features_harmonised': len(USABLE), 'pool': len(POOL), 'reference_relative_change': REF_CHANGE},
           'robust': {'n_robust': len(ROBUST), 'pool': len(POOL), 'threshold': ICC_THRESHOLD, 'cv_auc_mean': float(np.mean(cv_robust)),
                      'cv_auc_sd': float(np.std(cv_robust)), 'e2_cv_auc_mean': float(np.mean(cv_e2))},
           'consistency_with_E3': CONSISTENCY, 'hcc_tace_seg_check': TACE_CHECK,
           'protocol': '00 log section 21 (approved 17 Oct 2026, without E8d); evaluated once; E6 primary unchanged'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest_out = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)}
                for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest_out).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E8a_{RUN_ID} | internal check passed: {GATE_OK} | E8.0 restored {RESTORED} -> {len(ALL203)} patients; '
      f'consistency max diff {max(E80_CHECK.values()):.1e}; header agrees: {orientation_summary.get("header_agrees")}')
print(f"E8.0 PRIMARY on 203 (pre-specified, unchanged rule): CNN + radiomics [D-CNN] AUC {PRIM_E80['auc']:.3f} "
      f"({PRIM_E80['auc_delong_low']:.3f}-{PRIM_E80['auc_delong_high']:.3f}) | E6 run on 201: 0.581")
print(f'ComBat: {len(USABLE)}/{len(POOL)} features, reference change {REF_CHANGE:.1e} | robust features {len(ROBUST)}/{len(POOL)} '
      f'| internal CV radiomics E2 {np.mean(cv_e2):.3f} vs robust {np.mean(cv_robust):.3f} | consistency with E3: '
      f'{ {k: round(v, 3) for k, v in CONSISTENCY.items()} } | HCC-TACE-Seg check {TACE_CHECK:.1e}')
print('\n--- MCT-LTDiag 203, primary preprocessing (AUC with DeLong CI; argmax sens/spec) ---')
for _, r in MIT_RESULTS.iterrows():
    print(f"{r['model']:62s} AUC {r['auc']:.3f} ({r['auc_delong_low']:.3f}-{r['auc_delong_high']:.3f}) | sens {r['hcc_sensitivity']:.2f} "
          f"| spec {r['icc_specificity']:.2f}")
print('\n--- paired AUC differences vs the unmitigated model (bootstrap 95% CI, DeLong p) ---')
print(MIT_PAIRED.round(3).to_string(index=False))
print('\n--- internal held-out (56) macro-AUC ---')
print(INTERNAL.round(3).to_string(index=False))
print('\n--- HCC-TACE-Seg corrected (E5c geometry), HCC recognised ---')
print(TACE_RESULTS[TACE_RESULTS['subset'] == 'clean_89'][['model', 'hcc_correct', 'n', 'hcc_sensitivity']].round(3).to_string(index=False))
print('\nVersions:', VERSIONS)
print('==============================================================')